In [1]:
import pandas as pd 
import numpy as np 
from  statsmodels.stats  import stattools
import plotly.graph_objects as go 
import plotly.express as px

## LR Assumptions
* The target and the predictor relationship is linear. 
* Residuals are independent of each other.
* Predictors are not highly correlated with each other.

In [2]:
house_price = pd.read_csv(r"C:\potfolio\Advance_house_price_prediction\data\raw\train.csv")
house_price.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1460 entries, 0 to 1459
Data columns (total 81 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Id             1460 non-null   int64  
 1   MSSubClass     1460 non-null   int64  
 2   MSZoning       1460 non-null   object 
 3   LotFrontage    1201 non-null   float64
 4   LotArea        1460 non-null   int64  
 5   Street         1460 non-null   object 
 6   Alley          91 non-null     object 
 7   LotShape       1460 non-null   object 
 8   LandContour    1460 non-null   object 
 9   Utilities      1460 non-null   object 
 10  LotConfig      1460 non-null   object 
 11  LandSlope      1460 non-null   object 
 12  Neighborhood   1460 non-null   object 
 13  Condition1     1460 non-null   object 
 14  Condition2     1460 non-null   object 
 15  BldgType       1460 non-null   object 
 16  HouseStyle     1460 non-null   object 
 17  OverallQual    1460 non-null   int64  
 18  OverallC

In [3]:
missing_list = {}
length = len(house_price)
for col in house_price.columns:
    missing = ((house_price[col].isna().sum())/length)*100
    missing_list[col] = round(missing,2)
missing_list

{'Id': 0.0,
 'MSSubClass': 0.0,
 'MSZoning': 0.0,
 'LotFrontage': 17.74,
 'LotArea': 0.0,
 'Street': 0.0,
 'Alley': 93.77,
 'LotShape': 0.0,
 'LandContour': 0.0,
 'Utilities': 0.0,
 'LotConfig': 0.0,
 'LandSlope': 0.0,
 'Neighborhood': 0.0,
 'Condition1': 0.0,
 'Condition2': 0.0,
 'BldgType': 0.0,
 'HouseStyle': 0.0,
 'OverallQual': 0.0,
 'OverallCond': 0.0,
 'YearBuilt': 0.0,
 'YearRemodAdd': 0.0,
 'RoofStyle': 0.0,
 'RoofMatl': 0.0,
 'Exterior1st': 0.0,
 'Exterior2nd': 0.0,
 'MasVnrType': 59.73,
 'MasVnrArea': 0.55,
 'ExterQual': 0.0,
 'ExterCond': 0.0,
 'Foundation': 0.0,
 'BsmtQual': 2.53,
 'BsmtCond': 2.53,
 'BsmtExposure': 2.6,
 'BsmtFinType1': 2.53,
 'BsmtFinSF1': 0.0,
 'BsmtFinType2': 2.6,
 'BsmtFinSF2': 0.0,
 'BsmtUnfSF': 0.0,
 'TotalBsmtSF': 0.0,
 'Heating': 0.0,
 'HeatingQC': 0.0,
 'CentralAir': 0.0,
 'Electrical': 0.07,
 '1stFlrSF': 0.0,
 '2ndFlrSF': 0.0,
 'LowQualFinSF': 0.0,
 'GrLivArea': 0.0,
 'BsmtFullBath': 0.0,
 'BsmtHalfBath': 0.0,
 'FullBath': 0.0,
 'HalfBath': 0.0,

In [4]:
selected_columns = []
resonable_missing = []
for col_name, missing in missing_list.items():
    if missing > 0 and missing < 10:
        resonable_missing.append(col_name)
        continue
    if  missing < 15:
        selected_columns.append(col_name)
print(len(missing_list))
print(len(selected_columns))
resonable_missing

81
62


['MasVnrArea',
 'BsmtQual',
 'BsmtCond',
 'BsmtExposure',
 'BsmtFinType1',
 'BsmtFinType2',
 'Electrical',
 'GarageType',
 'GarageYrBlt',
 'GarageFinish',
 'GarageQual',
 'GarageCond']

In [5]:
df = house_price[selected_columns+resonable_missing]
df.shape

(1460, 74)

In [6]:
def impute_missing(col):
    return house_price[col].mean()

In [7]:
for col in resonable_missing:
    print(df[col].dtype)

float64
object
object
object
object
object
object
object
float64
object
object
object


In [8]:
for col in resonable_missing:
    type = df[col].dtype
    if type == "float64" :
        fill = df[col].mean()
        df[col] = df[col].fillna(fill)
    else:
        df[col] = df[col].fillna("unknown")
df.head()

C:\Users\RITHIK KUMAR\AppData\Local\Temp\ipykernel_17768\2237177990.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df[col] = df[col].fillna(fill)
C:\Users\RITHIK KUMAR\AppData\Local\Temp\ipykernel_17768\2237177990.py:7: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df[col] = df[col].fillna("unknown")


,Id,MSSubClass,MSZoning,LotArea,Street,LotShape,LandContour,Utilities,LotConfig,LandSlope,...,BsmtCond,BsmtExposure,BsmtFinType1,BsmtFinType2,Electrical,GarageType,GarageYrBlt,GarageFinish,GarageQual,GarageCond
0,1,60,RL,8450,Pave,Reg,Lvl,AllPub,Inside,Gtl,...,TA,No,GLQ,Unf,SBrkr,Attchd,2003.0,RFn,TA,TA
1,2,20,RL,9600,Pave,Reg,Lvl,AllPub,FR2,Gtl,...,TA,Gd,ALQ,Unf,SBrkr,Attchd,1976.0,RFn,TA,TA
2,3,60,RL,11250,Pave,IR1,Lvl,AllPub,Inside,Gtl,...,TA,Mn,GLQ,Unf,SBrkr,Attchd,2001.0,RFn,TA,TA
3,4,70,RL,9550,Pave,IR1,Lvl,AllPub,Corner,Gtl,...,Gd,No,ALQ,Unf,SBrkr,Detchd,1998.0,Unf,TA,TA
4,5,60,RL,14260,Pave,IR1,Lvl,AllPub,FR2,Gtl,...,TA,Av,GLQ,Unf,SBrkr,Attchd,2000.0,RFn,TA,TA


In [9]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1460 entries, 0 to 1459
Data columns (total 74 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Id             1460 non-null   int64  
 1   MSSubClass     1460 non-null   int64  
 2   MSZoning       1460 non-null   object 
 3   LotArea        1460 non-null   int64  
 4   Street         1460 non-null   object 
 5   LotShape       1460 non-null   object 
 6   LandContour    1460 non-null   object 
 7   Utilities      1460 non-null   object 
 8   LotConfig      1460 non-null   object 
 9   LandSlope      1460 non-null   object 
 10  Neighborhood   1460 non-null   object 
 11  Condition1     1460 non-null   object 
 12  Condition2     1460 non-null   object 
 13  BldgType       1460 non-null   object 
 14  HouseStyle     1460 non-null   object 
 15  OverallQual    1460 non-null   int64  
 16  OverallCond    1460 non-null   int64  
 17  YearBuilt      1460 non-null   int64  
 18  YearRemo

In [10]:
df.isna().sum()

Id              0
MSSubClass      0
MSZoning        0
LotArea         0
Street          0
               ..
GarageType      0
GarageYrBlt     0
GarageFinish    0
GarageQual      0
GarageCond      0
Length: 74, dtype: int64

In [11]:
df.columns

Index(['Id', 'MSSubClass', 'MSZoning', 'LotArea', 'Street', 'LotShape',
       'LandContour', 'Utilities', 'LotConfig', 'LandSlope', 'Neighborhood',
       'Condition1', 'Condition2', 'BldgType', 'HouseStyle', 'OverallQual',
       'OverallCond', 'YearBuilt', 'YearRemodAdd', 'RoofStyle', 'RoofMatl',
       'Exterior1st', 'Exterior2nd', 'ExterQual', 'ExterCond', 'Foundation',
       'BsmtFinSF1', 'BsmtFinSF2', 'BsmtUnfSF', 'TotalBsmtSF', 'Heating',
       'HeatingQC', 'CentralAir', '1stFlrSF', '2ndFlrSF', 'LowQualFinSF',
       'GrLivArea', 'BsmtFullBath', 'BsmtHalfBath', 'FullBath', 'HalfBath',
       'BedroomAbvGr', 'KitchenAbvGr', 'KitchenQual', 'TotRmsAbvGrd',
       'Functional', 'Fireplaces', 'GarageCars', 'GarageArea', 'PavedDrive',
       'WoodDeckSF', 'OpenPorchSF', 'EnclosedPorch', '3SsnPorch',
       'ScreenPorch', 'PoolArea', 'MiscVal', 'MoSold', 'YrSold', 'SaleType',
       'SaleCondition', 'SalePrice', 'MasVnrArea', 'BsmtQual', 'BsmtCond',
       'BsmtExposure', 'BsmtFinTy

In [12]:
numerical_features = df.select_dtypes(include=["float64","int64"]).columns
num_df = df[numerical_features]
corr = num_df.corr()

fig = go.Figure()

fig.add_trace(
    go.Heatmap(
        z = corr.values,
        y = corr.columns,
        x = corr.columns,
        colorscale="RdBu",
        zmin=-1,
        zmax=1
    )
)
fig.update_layout(
    title="Correlation Heatmap",
    xaxis_title="Features",
    yaxis_title="Features",
    height=800
)

fig.show()

In [13]:
# import seaborn as sns
# import matplotlib.pyplot  as plt

# sns.pairplot(
#     df[numerical_features],
#     hue="SalePrice"
# )
# plt.show()

In [14]:
skewness = {} 
for col in numerical_features:
    skew = df[col].skew()
    skewness[col] = [skew]
    kurtosis = df[col].kurtosis()
    skewness[col].append(kurtosis)
skewness


{'Id': [0.0, -1.1999999999999997],
 'MSSubClass': [1.4076567471495591, 1.5801879649863309],
 'LotArea': [12.207687851233496, 203.24327101886033],
 'OverallQual': [0.2169439277628693, 0.09629277835615113],
 'OverallCond': [0.6930674724842182, 1.1064134613731684],
 'YearBuilt': [-0.613461172488183, -0.43955194159361977],
 'YearRemodAdd': [-0.5035620027004709, -1.2722451924732956],
 'BsmtFinSF1': [1.685503071910789, 11.118236291964712],
 'BsmtFinSF2': [4.255261108933303, 20.11333754558646],
 'BsmtUnfSF': [0.9202684528039037, 0.47499398780908475],
 'TotalBsmtSF': [1.5242545490627664, 13.250483281984796],
 '1stFlrSF': [1.3767566220336365, 5.74584148244079],
 '2ndFlrSF': [0.8130298163023265, -0.5534635576075795],
 'LowQualFinSF': [9.011341288465387, 83.2348166744174],
 'GrLivArea': [1.3665603560164552, 4.895120580693174],
 'BsmtFullBath': [0.596066609663168, -0.8390982654634271],
 'BsmtHalfBath': [4.103402697955168, 16.396641945350446],
 'FullBath': [0.036561558402727165, -0.8570428212743262

In [15]:
high_skewness = []
for key, value in skewness.items():
    if abs(value[0]) > 1:
        high_skewness.append(key)
high_skewness

['MSSubClass',
 'LotArea',
 'BsmtFinSF1',
 'BsmtFinSF2',
 'TotalBsmtSF',
 '1stFlrSF',
 'LowQualFinSF',
 'GrLivArea',
 'BsmtHalfBath',
 'KitchenAbvGr',
 'WoodDeckSF',
 'OpenPorchSF',
 'EnclosedPorch',
 '3SsnPorch',
 'ScreenPorch',
 'PoolArea',
 'MiscVal',
 'SalePrice',
 'MasVnrArea']

In [16]:
def plot_frequency(col):
    fig = go.Figure()
    fig.add_trace(
        go.Histogram(
            x=df[col],
            nbinsx=50,
            marker= dict(line =dict(width = 1, color ="black"))
        )
    )
    fig.update_layout(
        title=f"Distribution of {col}",
        xaxis_title=col,
        yaxis_title="Frequency",
        height=500
    )
    fig.add_vline(
    x=df[col].mean(),
    line_dash="dash",
    annotation_text="Mean"
    )
    
    fig.show()

In [17]:
for col in high_skewness:
    plot_frequency(col)

In [18]:
for col in high_skewness:
    df[col] = np.log1p(df[col])

C:\Users\RITHIK KUMAR\AppData\Local\Temp\ipykernel_17768\1544914593.py:2: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy



In [19]:
# After applying log tranformation lets visualize the graph again

for col in high_skewness:
    plot_frequency(col)

In [20]:
categorical_features = df.select_dtypes(include=["object"]).columns
categorical_features

Index(['MSZoning', 'Street', 'LotShape', 'LandContour', 'Utilities',
       'LotConfig', 'LandSlope', 'Neighborhood', 'Condition1', 'Condition2',
       'BldgType', 'HouseStyle', 'RoofStyle', 'RoofMatl', 'Exterior1st',
       'Exterior2nd', 'ExterQual', 'ExterCond', 'Foundation', 'Heating',
       'HeatingQC', 'CentralAir', 'KitchenQual', 'Functional', 'PavedDrive',
       'SaleType', 'SaleCondition', 'BsmtQual', 'BsmtCond', 'BsmtExposure',
       'BsmtFinType1', 'BsmtFinType2', 'Electrical', 'GarageType',
       'GarageFinish', 'GarageQual', 'GarageCond'],
      dtype='object')

In [21]:
df[categorical_features].head()

,MSZoning,Street,LotShape,LandContour,Utilities,LotConfig,LandSlope,Neighborhood,Condition1,Condition2,...,BsmtQual,BsmtCond,BsmtExposure,BsmtFinType1,BsmtFinType2,Electrical,GarageType,GarageFinish,GarageQual,GarageCond
0,RL,Pave,Reg,Lvl,AllPub,Inside,Gtl,CollgCr,Norm,Norm,...,Gd,TA,No,GLQ,Unf,SBrkr,Attchd,RFn,TA,TA
1,RL,Pave,Reg,Lvl,AllPub,FR2,Gtl,Veenker,Feedr,Norm,...,Gd,TA,Gd,ALQ,Unf,SBrkr,Attchd,RFn,TA,TA
2,RL,Pave,IR1,Lvl,AllPub,Inside,Gtl,CollgCr,Norm,Norm,...,Gd,TA,Mn,GLQ,Unf,SBrkr,Attchd,RFn,TA,TA
3,RL,Pave,IR1,Lvl,AllPub,Corner,Gtl,Crawfor,Norm,Norm,...,TA,Gd,No,ALQ,Unf,SBrkr,Detchd,Unf,TA,TA
4,RL,Pave,IR1,Lvl,AllPub,FR2,Gtl,NoRidge,Norm,Norm,...,Gd,TA,Av,GLQ,Unf,SBrkr,Attchd,RFn,TA,TA


In [22]:
for col in categorical_features:
    print(df[col].value_counts())

MSZoning
RL         1151
RM          218
FV           65
RH           16
C (all)      10
Name: count, dtype: int64
Street
Pave    1454
Grvl       6
Name: count, dtype: int64
LotShape
Reg    925
IR1    484
IR2     41
IR3     10
Name: count, dtype: int64
LandContour
Lvl    1311
Bnk      63
HLS      50
Low      36
Name: count, dtype: int64
Utilities
AllPub    1459
NoSeWa       1
Name: count, dtype: int64
LotConfig
Inside     1052
Corner      263
CulDSac      94
FR2          47
FR3           4
Name: count, dtype: int64
LandSlope
Gtl    1382
Mod      65
Sev      13
Name: count, dtype: int64
Neighborhood
NAmes      225
CollgCr    150
OldTown    113
Edwards    100
Somerst     86
Gilbert     79
NridgHt     77
Sawyer      74
NWAmes      73
SawyerW     59
BrkSide     58
Crawfor     51
Mitchel     49
NoRidge     41
Timber      38
IDOTRR      37
ClearCr     28
StoneBr     25
SWISU       25
MeadowV     17
Blmngtn     17
BrDale      16
Veenker     11
NPkVill      9
Blueste      2
Name: count, dtype:

In [23]:
drop_cols = ["Utilities", "Street", "Condition2"]
categorical_features = categorical_features.drop(drop_cols)
categorical_features

Index(['MSZoning', 'LotShape', 'LandContour', 'LotConfig', 'LandSlope',
       'Neighborhood', 'Condition1', 'BldgType', 'HouseStyle', 'RoofStyle',
       'RoofMatl', 'Exterior1st', 'Exterior2nd', 'ExterQual', 'ExterCond',
       'Foundation', 'Heating', 'HeatingQC', 'CentralAir', 'KitchenQual',
       'Functional', 'PavedDrive', 'SaleType', 'SaleCondition', 'BsmtQual',
       'BsmtCond', 'BsmtExposure', 'BsmtFinType1', 'BsmtFinType2',
       'Electrical', 'GarageType', 'GarageFinish', 'GarageQual', 'GarageCond'],
      dtype='object')

In [24]:
df[categorical_features]

,MSZoning,LotShape,LandContour,LotConfig,LandSlope,Neighborhood,Condition1,BldgType,HouseStyle,RoofStyle,...,BsmtQual,BsmtCond,BsmtExposure,BsmtFinType1,BsmtFinType2,Electrical,GarageType,GarageFinish,GarageQual,GarageCond
0,RL,Reg,Lvl,Inside,Gtl,CollgCr,Norm,1Fam,2Story,Gable,...,Gd,TA,No,GLQ,Unf,SBrkr,Attchd,RFn,TA,TA
1,RL,Reg,Lvl,FR2,Gtl,Veenker,Feedr,1Fam,1Story,Gable,...,Gd,TA,Gd,ALQ,Unf,SBrkr,Attchd,RFn,TA,TA
2,RL,IR1,Lvl,Inside,Gtl,CollgCr,Norm,1Fam,2Story,Gable,...,Gd,TA,Mn,GLQ,Unf,SBrkr,Attchd,RFn,TA,TA
3,RL,IR1,Lvl,Corner,Gtl,Crawfor,Norm,1Fam,2Story,Gable,...,TA,Gd,No,ALQ,Unf,SBrkr,Detchd,Unf,TA,TA
4,RL,IR1,Lvl,FR2,Gtl,NoRidge,Norm,1Fam,2Story,Gable,...,Gd,TA,Av,GLQ,Unf,SBrkr,Attchd,RFn,TA,TA
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1455,RL,Reg,Lvl,Inside,Gtl,Gilbert,Norm,1Fam,2Story,Gable,...,Gd,TA,No,Unf,Unf,SBrkr,Attchd,RFn,TA,TA
1456,RL,Reg,Lvl,Inside,Gtl,NWAmes,Norm,1Fam,1Story,Gable,...,Gd,TA,No,ALQ,Rec,SBrkr,Attchd,Unf,TA,TA
1457,RL,Reg,Lvl,Inside,Gtl,Crawfor,Norm,1Fam,2Story,Gable,...,TA,Gd,No,GLQ,Unf,SBrkr,Attchd,RFn,TA,TA
1458,RL,Reg,Lvl,Inside,Gtl,NAmes,Norm,1Fam,1Story,Hip,...,TA,TA,Mn,GLQ,Rec,FuseA,Attchd,Unf,TA,TA


In [25]:
arr = ["CentralAir",

"PavedDrive" ,

"GarageFinish",

"BsmtExposure"]

for col in arr:
    print(df[col].value_counts())

CentralAir
Y    1365
N      95
Name: count, dtype: int64
PavedDrive
Y    1340
N      90
P      30
Name: count, dtype: int64
GarageFinish
Unf        605
RFn        422
Fin        352
unknown     81
Name: count, dtype: int64
BsmtExposure
No         953
Av         221
Gd         134
Mn         114
unknown     38
Name: count, dtype: int64


In [26]:
df["CentralAir"] = df["CentralAir"].map({"Y" : 1, "N" : 0})
df["PavedDrive"] = df["PavedDrive"].map({"Y" : 1, "N" : 0,"P" : -1})
df["GarageFinish"] = df["GarageFinish"].map({"Unf" : 0, "RFn" : 1, "Fin" : 2, "unknown" : -1})
df["BsmtExposure"] = df["BsmtExposure"].map({"No" : 0, "Av" : 1, "Gd" : 2, "Mn" : 3 , "unknown" : -1})
df[arr].head()

C:\Users\RITHIK KUMAR\AppData\Local\Temp\ipykernel_17768\1185846802.py:1: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

C:\Users\RITHIK KUMAR\AppData\Local\Temp\ipykernel_17768\1185846802.py:2: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

C:\Users\RITHIK KUMAR\AppData\Local\Temp\ipykernel_17768\1185846802.py:3: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pa

,CentralAir,PavedDrive,GarageFinish,BsmtExposure
0,1,1,1,0
1,1,1,1,2
2,1,1,1,3
3,1,1,0,0
4,1,1,1,1


In [27]:
pd.set_option("display.max_columns",None)

In [28]:
df.head()

,Id,MSSubClass,MSZoning,LotArea,Street,LotShape,LandContour,Utilities,LotConfig,LandSlope,Neighborhood,Condition1,Condition2,BldgType,HouseStyle,OverallQual,OverallCond,YearBuilt,YearRemodAdd,RoofStyle,RoofMatl,Exterior1st,Exterior2nd,ExterQual,ExterCond,Foundation,BsmtFinSF1,BsmtFinSF2,BsmtUnfSF,TotalBsmtSF,Heating,HeatingQC,CentralAir,1stFlrSF,2ndFlrSF,LowQualFinSF,GrLivArea,BsmtFullBath,BsmtHalfBath,FullBath,HalfBath,BedroomAbvGr,KitchenAbvGr,KitchenQual,TotRmsAbvGrd,Functional,Fireplaces,GarageCars,GarageArea,PavedDrive,WoodDeckSF,OpenPorchSF,EnclosedPorch,3SsnPorch,ScreenPorch,PoolArea,MiscVal,MoSold,YrSold,SaleType,SaleCondition,SalePrice,MasVnrArea,BsmtQual,BsmtCond,BsmtExposure,BsmtFinType1,BsmtFinType2,Electrical,GarageType,GarageYrBlt,GarageFinish,GarageQual,GarageCond
0,1,4.110874,RL,9.042040,Pave,Reg,Lvl,AllPub,Inside,Gtl,CollgCr,Norm,Norm,1Fam,2Story,7,5,2003,2003,Gable,CompShg,VinylSd,VinylSd,Gd,TA,PConc,6.561031,0.0,150,6.753438,GasA,Ex,1,6.753438,854,0.0,7.444833,1,0.000000,2,1,3,0.693147,Gd,8,Typ,0,2,548,1,0.000000,4.127134,0.000000,0.0,0.0,0.0,0.0,2,2008,WD,Normal,12.247699,5.283204,Gd,TA,0,GLQ,Unf,SBrkr,Attchd,2003.0,1,TA,TA
1,2,3.044522,RL,9.169623,Pave,Reg,Lvl,AllPub,FR2,Gtl,Veenker,Feedr,Norm,1Fam,1Story,6,8,1976,1976,Gable,CompShg,MetalSd,MetalSd,TA,TA,CBlock,6.886532,0.0,284,7.141245,GasA,Ex,1,7.141245,0,0.0,7.141245,0,0.693147,2,0,3,0.693147,TA,6,Typ,1,2,460,1,5.700444,0.000000,0.000000,0.0,0.0,0.0,0.0,5,2007,WD,Normal,12.109016,0.000000,Gd,TA,2,ALQ,Unf,SBrkr,Attchd,1976.0,1,TA,TA
2,3,4.110874,RL,9.328212,Pave,IR1,Lvl,AllPub,Inside,Gtl,CollgCr,Norm,Norm,1Fam,2Story,7,5,2001,2002,Gable,CompShg,VinylSd,VinylSd,Gd,TA,PConc,6.188264,0.0,434,6.825460,GasA,Ex,1,6.825460,866,0.0,7.488294,1,0.000000,2,1,3,0.693147,Gd,6,Typ,1,2,608,1,0.000000,3.761200,0.000000,0.0,0.0,0.0,0.0,9,2008,WD,Normal,12.317171,5.093750,Gd,TA,3,GLQ,Unf,SBrkr,Attchd,2001.0,1,TA,TA
3,4,4.262680,RL,9.164401,Pave,IR1,Lvl,AllPub,Corner,Gtl,Crawfor,Norm,Norm,1Fam,2Story,7,5,1915,1970,Gable,CompShg,Wd Sdng,Wd Shng,TA,TA,BrkTil,5.379897,0.0,540,6.629363,GasA,Gd,1,6.869014,756,0.0,7.448916,1,0.000000,1,0,3,0.693147,Gd,7,Typ,1,3,642,1,0.000000,3.583519,5.609472,0.0,0.0,0.0,0.0,2,2006,WD,Abnorml,11.849405,0.000000,TA,Gd,0,ALQ,Unf,SBrkr,Detchd,1998.0,0,TA,TA
4,5,4.110874,RL,9.565284,Pave,IR1,Lvl,AllPub,FR2,Gtl,NoRidge,Norm,Norm,1Fam,2Story,8,5,2000,2000,Gable,CompShg,VinylSd,VinylSd,Gd,TA,PConc,6.486161,0.0,490,7.044033,GasA,Ex,1,7.044033,1053,0.0,7.695758,1,0.000000,2,1,4,0.693147,Gd,9,Typ,1,3,836,1,5.262690,4.442651,0.000000,0.0,0.0,0.0,0.0,12,2008,WD,Normal,12.429220,5.860786,Gd,TA,1,GLQ,Unf,SBrkr,Attchd,2000.0,1,TA,TA


In [29]:
arr1=["ExterQual",

"KitchenQual",

"HeatingQC",

"BsmtQual",

"GarageQual"]
for col in arr1:
    print(df[col].value_counts())

ExterQual
TA    906
Gd    488
Ex     52
Fa     14
Name: count, dtype: int64
KitchenQual
TA    735
Gd    586
Ex    100
Fa     39
Name: count, dtype: int64
HeatingQC
Ex    741
TA    428
Gd    241
Fa     49
Po      1
Name: count, dtype: int64
BsmtQual
TA         649
Gd         618
Ex         121
unknown     37
Fa          35
Name: count, dtype: int64
GarageQual
TA         1311
unknown      81
Fa           48
Gd           14
Ex            3
Po            3
Name: count, dtype: int64


In [30]:
qual_map = {
    "Po":1,
    "Fa":2,
    "TA":3,
    "Gd":4,
    "Ex":5,
    "unknown":0
}
for col in arr1:
    df[col] = df[col].map(qual_map)

df[arr1].head()

C:\Users\RITHIK KUMAR\AppData\Local\Temp\ipykernel_17768\2800577979.py:10: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

C:\Users\RITHIK KUMAR\AppData\Local\Temp\ipykernel_17768\2800577979.py:10: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

C:\Users\RITHIK KUMAR\AppData\Local\Temp\ipykernel_17768\2800577979.py:10: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https:/

,ExterQual,KitchenQual,HeatingQC,BsmtQual,GarageQual
0,4,4,5,4,3
1,3,3,5,4,3
2,4,4,5,4,3
3,3,4,4,3,3
4,4,4,5,4,3


In [31]:
arr2 = ["MSZoning",

"LotShape",

"LandContour",

"BldgType",

"HouseStyle",

"RoofStyle",

"SaleCondition"]

for col in arr2:
    print(df[col].value_counts())

MSZoning
RL         1151
RM          218
FV           65
RH           16
C (all)      10
Name: count, dtype: int64
LotShape
Reg    925
IR1    484
IR2     41
IR3     10
Name: count, dtype: int64
LandContour
Lvl    1311
Bnk      63
HLS      50
Low      36
Name: count, dtype: int64
BldgType
1Fam      1220
TwnhsE     114
Duplex      52
Twnhs       43
2fmCon      31
Name: count, dtype: int64
HouseStyle
1Story    726
2Story    445
1.5Fin    154
SLvl       65
SFoyer     37
1.5Unf     14
2.5Unf     11
2.5Fin      8
Name: count, dtype: int64
RoofStyle
Gable      1141
Hip         286
Flat         13
Gambrel      11
Mansard       7
Shed          2
Name: count, dtype: int64
SaleCondition
Normal     1198
Partial     125
Abnorml     101
Family       20
Alloca       12
AdjLand       4
Name: count, dtype: int64


In [32]:
df = pd.get_dummies(df, columns=arr2, drop_first=True)
df.head()

,Id,MSSubClass,LotArea,Street,Utilities,LotConfig,LandSlope,Neighborhood,Condition1,Condition2,OverallQual,OverallCond,YearBuilt,YearRemodAdd,RoofMatl,Exterior1st,Exterior2nd,ExterQual,ExterCond,Foundation,BsmtFinSF1,BsmtFinSF2,BsmtUnfSF,TotalBsmtSF,Heating,HeatingQC,CentralAir,1stFlrSF,2ndFlrSF,LowQualFinSF,GrLivArea,BsmtFullBath,BsmtHalfBath,FullBath,HalfBath,BedroomAbvGr,KitchenAbvGr,KitchenQual,TotRmsAbvGrd,Functional,Fireplaces,GarageCars,GarageArea,PavedDrive,WoodDeckSF,OpenPorchSF,EnclosedPorch,3SsnPorch,ScreenPorch,PoolArea,MiscVal,MoSold,YrSold,SaleType,SalePrice,MasVnrArea,BsmtQual,BsmtCond,BsmtExposure,BsmtFinType1,BsmtFinType2,Electrical,GarageType,GarageYrBlt,GarageFinish,GarageQual,GarageCond,MSZoning_FV,MSZoning_RH,MSZoning_RL,MSZoning_RM,LotShape_IR2,LotShape_IR3,LotShape_Reg,LandContour_HLS,LandContour_Low,LandContour_Lvl,BldgType_2fmCon,BldgType_Duplex,BldgType_Twnhs,BldgType_TwnhsE,HouseStyle_1.5Unf,HouseStyle_1Story,HouseStyle_2.5Fin,HouseStyle_2.5Unf,HouseStyle_2Story,HouseStyle_SFoyer,HouseStyle_SLvl,RoofStyle_Gable,RoofStyle_Gambrel,RoofStyle_Hip,RoofStyle_Mansard,RoofStyle_Shed,SaleCondition_AdjLand,SaleCondition_Alloca,SaleCondition_Family,SaleCondition_Normal,SaleCondition_Partial
0,1,4.110874,9.042040,Pave,AllPub,Inside,Gtl,CollgCr,Norm,Norm,7,5,2003,2003,CompShg,VinylSd,VinylSd,4,TA,PConc,6.561031,0.0,150,6.753438,GasA,5,1,6.753438,854,0.0,7.444833,1,0.000000,2,1,3,0.693147,4,8,Typ,0,2,548,1,0.000000,4.127134,0.000000,0.0,0.0,0.0,0.0,2,2008,WD,12.247699,5.283204,4,TA,0,GLQ,Unf,SBrkr,Attchd,2003.0,1,3,TA,False,False,True,False,False,False,True,False,False,True,False,False,False,False,False,False,False,False,True,False,False,True,False,False,False,False,False,False,False,True,False
1,2,3.044522,9.169623,Pave,AllPub,FR2,Gtl,Veenker,Feedr,Norm,6,8,1976,1976,CompShg,MetalSd,MetalSd,3,TA,CBlock,6.886532,0.0,284,7.141245,GasA,5,1,7.141245,0,0.0,7.141245,0,0.693147,2,0,3,0.693147,3,6,Typ,1,2,460,1,5.700444,0.000000,0.000000,0.0,0.0,0.0,0.0,5,2007,WD,12.109016,0.000000,4,TA,2,ALQ,Unf,SBrkr,Attchd,1976.0,1,3,TA,False,False,True,False,False,False,True,False,False,True,False,False,False,False,False,True,False,False,False,False,False,True,False,False,False,False,False,False,False,True,False
2,3,4.110874,9.328212,Pave,AllPub,Inside,Gtl,CollgCr,Norm,Norm,7,5,2001,2002,CompShg,VinylSd,VinylSd,4,TA,PConc,6.188264,0.0,434,6.825460,GasA,5,1,6.825460,866,0.0,7.488294,1,0.000000,2,1,3,0.693147,4,6,Typ,1,2,608,1,0.000000,3.761200,0.000000,0.0,0.0,0.0,0.0,9,2008,WD,12.317171,5.093750,4,TA,3,GLQ,Unf,SBrkr,Attchd,2001.0,1,3,TA,False,False,True,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,True,False,False,True,False,False,False,False,False,False,False,True,False
3,4,4.262680,9.164401,Pave,AllPub,Corner,Gtl,Crawfor,Norm,Norm,7,5,1915,1970,CompShg,Wd Sdng,Wd Shng,3,TA,BrkTil,5.379897,0.0,540,6.629363,GasA,4,1,6.869014,756,0.0,7.448916,1,0.000000,1,0,3,0.693147,4,7,Typ,1,3,642,1,0.000000,3.583519,5.609472,0.0,0.0,0.0,0.0,2,2006,WD,11.849405,0.000000,3,Gd,0,ALQ,Unf,SBrkr,Detchd,1998.0,0,3,TA,False,False,True,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,True,False,False,True,False,False,False,False,False,False,False,False,False
4,5,4.110874,9.565284,Pave,AllPub,FR2,Gtl,NoRidge,Norm,Norm,8,5,2000,2000,CompShg,VinylSd,VinylSd,4,TA,PConc,6.486161,0.0,490,7.044033,GasA,5,1,7.044033,1053,0.0,7.695758,1,0.000000,2,1,4,0.693147,4,9,Typ,1,3,836,1,5.262690,4.442651,0.000000,0.0,0.0,0.0,0.0,12,2008,WD,12.429220,5.860786,4,TA,1,GLQ,Unf,SBrkr,Attchd,2000.0,1,3,TA,False,False,True,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,True,False,False,True,False,False,False,False,False,False,False,True,False


In [33]:
new_cat_features = df.select_dtypes(include=["object"]).columns
df[new_cat_features][456:480]

,Street,Utilities,LotConfig,LandSlope,Neighborhood,Condition1,Condition2,RoofMatl,Exterior1st,Exterior2nd,ExterCond,Foundation,Heating,Functional,SaleType,BsmtCond,BsmtFinType1,BsmtFinType2,Electrical,GarageType,GarageCond
456,Pave,AllPub,Inside,Gtl,OldTown,Norm,Norm,CompShg,AsbShng,AsbShng,TA,BrkTil,GasA,Typ,COD,TA,Unf,Unf,SBrkr,Detchd,Fa
457,Pave,AllPub,CulDSac,Mod,ClearCr,Norm,Norm,Tar&Grv,Plywood,Plywood,TA,CBlock,GasA,Min1,WD,TA,BLQ,Unf,SBrkr,Attchd,TA
458,Pave,AllPub,Inside,Gtl,OldTown,Norm,Norm,CompShg,Stucco,Wd Shng,Gd,PConc,GasA,Typ,WD,TA,Unf,Unf,SBrkr,Detchd,TA
459,Pave,AllPub,Corner,Gtl,BrkSide,Norm,Norm,CompShg,MetalSd,MetalSd,TA,CBlock,GasA,Typ,WD,TA,LwQ,Unf,SBrkr,Detchd,TA
460,Pave,AllPub,Inside,Gtl,Somerst,RRAn,Norm,CompShg,VinylSd,VinylSd,TA,PConc,GasA,Typ,New,TA,GLQ,Unf,SBrkr,BuiltIn,TA
461,Pave,AllPub,Inside,Gtl,SWISU,Feedr,Norm,CompShg,Wd Sdng,Wd Sdng,Gd,PConc,GasA,Typ,WD,Gd,ALQ,BLQ,SBrkr,Detchd,TA
462,Pave,AllPub,Inside,Gtl,Sawyer,Norm,Norm,CompShg,MetalSd,MetalSd,TA,CBlock,GasA,Typ,WD,TA,Rec,BLQ,SBrkr,Detchd,TA
463,Pave,AllPub,Inside,Mod,Crawfor,Norm,Norm,CompShg,Stucco,Stucco,TA,CBlock,GasA,Typ,WD,TA,LwQ,Unf,FuseA,Detchd,TA
464,Pave,AllPub,Inside,Mod,CollgCr,Norm,Norm,CompShg,HdBoard,HdBoard,TA,CBlock,GasA,Typ,WD,TA,Rec,Unf,SBrkr,unknown,unknown
465,Pave,AllPub,Inside,Gtl,Blmngtn,Norm,Norm,CompShg,VinylSd,VinylSd,TA,PConc,GasA,Typ,WD,TA,Unf,Unf,SBrkr,Attchd,TA


In [34]:
df = df.drop(columns=drop_cols)
df.columns

Index(['Id', 'MSSubClass', 'LotArea', 'LotConfig', 'LandSlope', 'Neighborhood',
       'Condition1', 'OverallQual', 'OverallCond', 'YearBuilt', 'YearRemodAdd',
       'RoofMatl', 'Exterior1st', 'Exterior2nd', 'ExterQual', 'ExterCond',
       'Foundation', 'BsmtFinSF1', 'BsmtFinSF2', 'BsmtUnfSF', 'TotalBsmtSF',
       'Heating', 'HeatingQC', 'CentralAir', '1stFlrSF', '2ndFlrSF',
       'LowQualFinSF', 'GrLivArea', 'BsmtFullBath', 'BsmtHalfBath', 'FullBath',
       'HalfBath', 'BedroomAbvGr', 'KitchenAbvGr', 'KitchenQual',
       'TotRmsAbvGrd', 'Functional', 'Fireplaces', 'GarageCars', 'GarageArea',
       'PavedDrive', 'WoodDeckSF', 'OpenPorchSF', 'EnclosedPorch', '3SsnPorch',
       'ScreenPorch', 'PoolArea', 'MiscVal', 'MoSold', 'YrSold', 'SaleType',
       'SalePrice', 'MasVnrArea', 'BsmtQual', 'BsmtCond', 'BsmtExposure',
       'BsmtFinType1', 'BsmtFinType2', 'Electrical', 'GarageType',
       'GarageYrBlt', 'GarageFinish', 'GarageQual', 'GarageCond',
       'MSZoning_FV', 'MSZonin

In [35]:
import pandas as pd
import numpy as np
import joblib
from sklearn.preprocessing import StandardScaler

def preprocess_train(df):

    df = pd.read_csv(r"C:\potfolio\Advance_house_price_prediction\data\raw\train.csv")

    # Drop
    df.drop(columns=["Id"], inplace=True, errors="ignore")

    # -------- Ordinal Encoding --------
    qual_map = {"Po":1,"Fa":2,"TA":3,"Gd":4,"Ex":5,"unknown":0}
    bsmt_map = {"Unf":0,"LwQ":1,"Rec":2,"BLQ":3,"ALQ":4,"GLQ":5,"unknown":-1}
    func_map = {"Sev":1,"Maj2":2,"Maj1":3,"Mod":4,"Min2":5,"Min1":6,"Typ":7}

    for col in ["ExterCond","BsmtCond"]:
        if col in df.columns:
            df[col] = df[col].map(qual_map)

    if "BsmtFinType1" in df.columns:
        df["BsmtFinType1"] = df["BsmtFinType1"].map(bsmt_map)

    if "BsmtFinType2" in df.columns:
        df["BsmtFinType2"] = df["BsmtFinType2"].map(bsmt_map)

    if "Functional" in df.columns:
        df["Functional"] = df["Functional"].map(func_map)

    # -------- Target Encoding --------
    neigh_map = df.groupby("Neighborhood")["SalePrice"].mean()
    df["Neighborhood"] = df["Neighborhood"].map(neigh_map)

    # -------- Rare Handling --------
    freq = df["RoofMatl"].value_counts()
    rare = freq[freq < 10].index
    df["RoofMatl"] = df["RoofMatl"].replace(rare,"Other")

    # -------- Combine Exterior --------
    df["Exterior"] = df["Exterior1st"] + "_" + df["Exterior2nd"]
    df.drop(columns=["Exterior1st","Exterior2nd"], inplace=True)

    # -------- One-hot --------
    df = pd.get_dummies(df, drop_first=True)

    # -------- Separate --------
    y = np.log1p(df["SalePrice"])
    X = df.drop(columns=["SalePrice"])

    # -------- Scale --------
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)

    # Save artifacts
    joblib.dump(scaler, "scaler.pkl")
    joblib.dump(neigh_map, "neighborhood_map.pkl")
    joblib.dump(X.columns, "train_columns.pkl")

    # Save processed train
    train_df = pd.DataFrame(X_scaled, columns=X.columns)
    train_df["SalePrice"] = y.values
    train_df.to_csv(r"C:\potfolio\Advance_house_price_prediction\data\processed_data\processed_train.csv", index=False)

    return train_df

In [36]:
print(preprocess_train(df))

      MSSubClass  LotFrontage   LotArea  Neighborhood  OverallQual  \
0       0.073375    -0.208034 -0.207142      0.290573     0.651479   
1      -0.872563     0.409895 -0.091886      0.986242    -0.071836   
2       0.073375    -0.084449  0.073480      0.290573     0.651479   
3       0.309859    -0.414011 -0.096897      0.506380     0.651479   
4       0.073375     0.574676  0.375148      2.631741     1.374795   
...          ...          ...       ...           ...          ...   
1455    0.073375    -0.331620 -0.260560      0.203437    -0.071836   
1456   -0.872563     0.615871  0.266407      0.138579    -0.071836   
1457    0.309859    -0.166839 -0.147810      0.506380     0.651479   
1458   -0.872563    -0.084449 -0.080160     -0.597937    -0.795151   
1459   -0.872563     0.203918 -0.058112     -0.898445    -0.795151   

      OverallCond  YearBuilt  YearRemodAdd  MasVnrArea  ExterCond  BsmtCond  \
0       -0.517200   1.050994      0.878668    0.510015  -0.238112 -0.039076   
1

In [37]:
processed_df = pd.read_csv(r"C:\potfolio\Advance_house_price_prediction\data\processed_data\processed_train.csv")

corr = processed_df.corr()


fig = go.Figure()

fig.add_trace(
    go.Heatmap(
        z = corr.values,
        y = corr.columns,
        x = corr.columns,
        colorscale="RdBu",
        zmin=-1,
        zmax=1
    )
)
fig.update_layout(
    title="Correlation Heatmap",
    xaxis_title="Features",
    yaxis_title="Features",
    height = 1000,
    width = 1500
)

fig.show()
